In [1]:
import os
from dotenv import load_dotenv
#langchain
from langchain_community.document_loaders import TextLoader

C:\Users\User\AppData\Local\Temp\ipykernel_11680\1203596522.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [2]:
load_dotenv()

True

In [3]:
groq_key=os.getenv("GROQ_API_KEY")
jina_key=os.getenv("JINA_API_KEY")
print("Environment variables loaded")


Environment variables loaded


LOAD DATA PATH

In [4]:
DATA_FILE_PATH=os.path.join("data","hr_policy.txt")

DATA INGESTION


In [5]:
loader=TextLoader(DATA_FILE_PATH,encoding="utf-8")
documents=loader.load()
print("DATA LOADED")
print("="*40)
print(documents)

DATA LOADED
[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)\n\n1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.\n\n2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM to 4 PM.\n\n3. PROBATION PERIOD\nAll new employees undergo a probation period of 3 months from their date of joining.\nDu

#### DOCUMENTS
 Has two parts
 1.meta data
 2.page content (actual data)

In [6]:
print(documents[0].page_content)

COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)

1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

2. WORK FROM HOME POLICY
Employees may work from home up to 2 days per week, subject to manager approval.
Fully remote work arrangements require written approval from the department head.
Employees working from home must be reachable during core hours: 10 AM to 4 PM.

3. PROBATION PERIOD
All new employees undergo a probation period of 3 months from their date of joining.
During probation, employees are not eligible for paid leave, but may take unpaid leave
in case of e

### SPLITTING

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
    )
chunks=text_splitter.split_documents(documents)
print(chunks[0])

page_content='COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)' metadata={'source': 'data\\hr_policy.txt'}


## MAKING EMBEDDINGS


In [8]:
from langchain_community.embeddings import JinaEmbeddings
embeddings_model = JinaEmbeddings(model_name="jina-embeddings-v2-base-en")

print("EMB MODEL READY THE NAME IS ", embeddings_model.model_name)

EMB MODEL READY THE NAME IS  jina-embeddings-v2-base-en


### STORE DATA IN VECTOR DB

In [9]:
from langchain_community.vectorstores import FAISS 

vector_store = FAISS.from_documents(chunks , embeddings_model)

print("CHUNKS ARE STORED" , vector_store.index.ntotal)

CHUNKS ARE STORED 9


### SIMILARTIY SEARCH

In [10]:
test_query = "How many sick leaves employees get"

## SIMILARITY SEARCH 

top_matches = vector_store.similarity_search(test_query , k=2)
print(f"Query: {test_query}\n")
for i,match in enumerate(top_matches,start=1):
    print(f"--- Match {i} ---")
    print(match.page_content)
    print()

Query: How many sick leaves employees get

--- Match 1 ---
1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

--- Match 2 ---
7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.



### DATA RETRIEVAL


In [ ]:
from langchain_groq import ChatGroq
llm=ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.5,
    stream=True
)
llm.model_name

'openai/gpt-oss-120b'

In [12]:
tr = llm.invoke("Hey what is the leave policy")
tr.content

'Sure thing! Here’s a quick rundown of our company’s leave policy. If you need more detail on any specific type of leave or have a unique situation, just let me know and I can dive deeper.\n\n---\n\n## 1. Annual / Vacation Leave\n| **Eligibility** | **Accrual Rate** | **Maximum Balance** | **How to Use** |\n|-----------------|------------------|---------------------|----------------|\n| All full‑time employees (after 3\u202fmonths of service) | 1.5\u202fdays per month (≈\u202f18\u202fdays per year) | 30\u202fdays (carries over up to 15\u202fdays to the next year) | Submit a request via the HR portal at least **5\u202fbusiness days** before the start date (unless it’s an emergency). Manager approval is required. |\n\n*Part‑time staff accrue on a pro‑rated basis.*\n\n---\n\n## 2. Sick Leave\n- **Accrual:** 1\u202fday per month (12\u202fdays per year).  \n- **Carry‑over:** Up to 10\u202fdays can be carried over to the next calendar year; any unused sick days beyond that expire.  \n- **Usa

### AI AGENT
An AI agent mainly has two parts the Brain which is an llm and the tools

#### TOOL

In [13]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})  # returns top 3 relevant chunks

def search_hr_policy(question:str)->str:
    """
    Search the HR policy document for information about leave, work from home,
    probation, notice period, reimbursement, code of conduct, holidays, or exit process.
    
    """
    matching_chunks = retriever.invoke(question)
    return "\n\n".join(chunk.page_content for chunk in matching_chunks)

Now we will create agent

In [14]:
from langchain.agents import create_agent

In [18]:
hr_assistant=create_agent(
    model=llm,
    tools=[search_hr_policy],
       system_prompt= """ 
    
    You are a friendly HR assistant working for Acme Crop. 
    Always use the search_hr_policy tool to look up 
    facts before answering. 
    If the answer isn't in the search results, say you don't know "
    instead of guessing."
    """
)
print("HR assistant agent is ready to answer questions!")

HR assistant agent is ready to answer questions!


In [16]:
def ask_hr_assistant(question: str) -> str:
    """Send a question to the RAG agent and print a nicely formatted answer."""
    print("=" * 60)
    print("QUESTION:", question)
    print("-" * 60)

    response = hr_assistant.invoke({"messages": [{"role": "user", "content": question}]})
    answer = response["messages"][-1].content

    print("ANSWER:", answer)
    print("=" * 60)
    print()
    return answer

In [23]:
response = hr_assistant.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content": "Can I leave the company by ghosting them?"
            }
        ]
    }
)

In [24]:
print(response["messages"][-1].content)

I’m afraid you can’t just “ghost” your way out of Acme Crop.  

Our HR policy requires that any resignation go through a formal process:

* **Notice period** – You must give 30 days’ notice (or 15 days if you’re still on probation) unless the company waives it in writing.  
* **Clearance** – Before your last working day you need to complete the clearance steps with IT, Finance and HR.  
* **Settlement** – Your final pay, any reimbursements and leave encashment are processed only after the clearance is finished (within 45 days of your last day).

Skipping the notice and clearance steps would be a breach of the company’s exit policy and could result in delayed or forfeited final settlement, as well as a negative record in your employment file.

So, if you’re thinking of leaving, the proper route is to submit a resignation letter, serve the required notice (or request a waiver), and complete the clearance process. If you need help drafting your resignation or understanding any part of the